In [2]:
import os
import warnings

# Hide warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import gradio as gr
import faiss

from sklearn.feature_extraction.text import TfidfVectorizer

# =====================================================
# DATASET PATHS
# =====================================================

DATASET_FOLDER = r"C:\Users\bhard\indian_drug_interaction_dataset"

INTERACTION_FILE = os.path.join(DATASET_FOLDER, "interactions.csv")
DRUG_INFO_FILE = os.path.join(DATASET_FOLDER, "drug_info.csv")
SIDE_EFFECT_FILE = os.path.join(DATASET_FOLDER, "side_effects.csv")

# =====================================================
# CHECK FILES
# =====================================================

required_files = [
    INTERACTION_FILE,
    DRUG_INFO_FILE,
    SIDE_EFFECT_FILE
]

for file in required_files:

    if not os.path.exists(file):

        raise FileNotFoundError(
            f"\nMissing File:\n{file}"
        )

# =====================================================
# LOAD DATA
# =====================================================

print("Loading datasets...")

interaction_df = pd.read_csv(INTERACTION_FILE)
drug_df = pd.read_csv(DRUG_INFO_FILE)
side_df = pd.read_csv(SIDE_EFFECT_FILE)

interaction_df.fillna("", inplace=True)
drug_df.fillna("", inplace=True)
side_df.fillna("", inplace=True)

# =====================================================
# VERIFY COLUMNS
# =====================================================

required_interaction_columns = [
    "drug_a",
    "drug_b",
    "severity",
    "description",
    "recommendation"
]

for col in required_interaction_columns:

    if col not in interaction_df.columns:

        raise Exception(
            f"Column '{col}' not found in interactions.csv"
        )

# =====================================================
# CREATE DOCUMENTS
# =====================================================

documents = []

for _, row in interaction_df.iterrows():

    text = (
        f"Drug A: {row['drug_a']} | "
        f"Drug B: {row['drug_b']} | "
        f"Severity: {row['severity']} | "
        f"Description: {row['description']} | "
        f"Recommendation: {row['recommendation']}"
    )

    documents.append(text)

print(f"Loaded {len(documents)} interaction records")

# =====================================================
# TF-IDF VECTORIZATION
# =====================================================

print("Creating vectors...")

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(documents)

embeddings = vectors.toarray().astype("float32")

# =====================================================
# FAISS INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS Index Ready")

# =====================================================
# RETRIEVE DOCUMENTS
# =====================================================

def retrieve(query, top_k=5):

    query_vector = vectorizer.transform(
        [query]
    )

    query_vector = query_vector.toarray().astype(
        "float32"
    )

    distances, indices = index.search(
        query_vector,
        min(top_k, len(documents))
    )

    results = []

    for idx in indices[0]:

        if idx < len(documents):

            results.append(
                documents[idx]
            )

    return results

# =====================================================
# DRUG INFO
# =====================================================

def get_drug_info(drug_name):

    if "drug_name" not in drug_df.columns:

        return "Drug info unavailable"

    rows = drug_df[
        drug_df["drug_name"]
        .astype(str)
        .str.lower()
        ==
        drug_name.lower()
    ]

    if len(rows) == 0:

        return "Drug info unavailable"

    row = rows.iloc[0]

    category = row.get(
        "category",
        "Unknown"
    )

    uses = row.get(
        "uses",
        "Unknown"
    )

    return (
        f"Category: {category}\n"
        f"Uses: {uses}"
    )

# =====================================================
# SIDE EFFECTS
# =====================================================

def get_side_effects(drug_name):

    if "drug_name" not in side_df.columns:

        return "No side-effect data"

    rows = side_df[
        side_df["drug_name"]
        .astype(str)
        .str.lower()
        ==
        drug_name.lower()
    ]

    if len(rows) == 0:

        return "No side-effect data"

    return ", ".join(
        rows["side_effect"]
        .astype(str)
        .tolist()
    )

# =====================================================
# RISK LEVEL
# =====================================================

def get_risk_level(text):

    text = text.lower()

    if "high" in text:

        return "🔴 HIGH RISK"

    if "moderate" in text:

        return "🟠 MODERATE RISK"

    return "🟢 LOW RISK"

# =====================================================
# MAIN ANALYSIS
# =====================================================

def analyze_drugs(user_input):

    if not user_input:

        return "Please enter drug names."

    results = retrieve(user_input)

    risk = get_risk_level(
        " ".join(results)
    )

    report = []

    report.append(
        "DRUG INTERACTION SAFETY REPORT"
    )

    report.append("=" * 50)

    report.append(
        f"\nOverall Risk: {risk}\n"
    )

    report.append(
        "\nInteraction Records:\n"
    )

    for item in results:

        report.append(item)

    report.append(
        "\n\nDrug Details\n"
    )

    drugs = [
        x.strip()
        for x in user_input.split(",")
    ]

    for drug in drugs:

        report.append(
            f"\nDrug: {drug}"
        )

        report.append(
            get_drug_info(drug)
        )

        report.append(
            f"Side Effects: {get_side_effects(drug)}"
        )

    if "HIGH" in risk:

        report.append(
            "\nRecommendation: Consult a physician immediately."
        )

    elif "MODERATE" in risk:

        report.append(
            "\nRecommendation: Use under supervision."
        )

    else:

        report.append(
            "\nRecommendation: No major interaction detected."
        )

    return "\n".join(report)

# =====================================================
# GRADIO APP
# =====================================================

demo = gr.Interface(
    fn=analyze_drugs,

    inputs=gr.Textbox(
        lines=2,
        label="Enter Drugs",
        placeholder="Warfarin,Aspirin"
    ),

    outputs=gr.Textbox(
        lines=30,
        label="Drug Safety Report"
    ),

    title="Drug Interaction Safety Assistant",

    description="Healthcare RAG System using TF-IDF + FAISS"
)

# =====================================================
# RUN
# =====================================================

if __name__ == "__main__":

    print("Starting Application...")

    demo.launch(
        share=False,
        debug=False,
        show_error=True
    )

Loading datasets...
Loaded 10 interaction records
Creating vectors...
FAISS Index Ready
Starting Application...
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
